# module-extra-repr — worked example 3: extra_repr above indented children in a stack

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `module-extra-repr`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When a parent module owns registered child modules, PyTorch prints the parent's `extra_repr` first, then each child's full repr indented two spaces beneath it. Overriding `extra_repr` on the parent does not disturb that recursion; it only adds a summary line for the parent itself.

## Worked solution

We build `TinyStack`, a parent owning two registered `nn.Linear` children, `proj_in` and `proj_out`, plus a `depth` integer we want surfaced. After `super().__init__()` we assign the children as attributes so they register, and store `depth`. The `extra_repr` returns just `depth=...`. When we print the model, PyTorch composes the tree: the top line is `TinyStack(depth=2)` (our string), and beneath it the `(proj_in): Linear(...)` and `(proj_out): Linear(...)` lines appear with two-space indentation, contributed automatically by `nn.Module.__repr__`. We print the repr to see both the parent summary and the indented children, demonstrating that our override coexists with the recursive child printing.

In [ ]:
import torch as t
import torch.nn as nn

class TinyStack(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, depth):
        super().__init__()
        self.depth = depth
        self.proj_in = nn.Linear(in_dim, hidden)
        self.proj_out = nn.Linear(hidden, out_dim)

    def extra_repr(self):
        return f'depth={self.depth}'

    def forward(self, x):
        return self.proj_out(t.relu(self.proj_in(x)))

model = TinyStack(10, 32, 4, depth=2)
print(repr(model))